# 04 · Evaluation, baselines & ablations

Produces the paper's results tables: the full model against three baselines on the held-out test
year, and the architectural ablations.

**All numbers are generated here by running the code on the real test split.** Nothing in this
notebook is hard-coded. If a model checkpoint is missing, its row is left blank — it is never filled
with a placeholder.

In [ ]:
import os, sys, pickle, json
from pathlib import Path
import numpy as np, pandas as pd, torch

sys.path.insert(0, os.path.abspath('..'))
from src.utils import load_config
from src.evaluate import evaluate_model, predict_split, print_table, save_results
from src.models.multimodal_transformer import MultimodalEnergyTransformer
from src.baselines import run_baselines

cfg = load_config('../configs/default.yaml')
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

with open(Path(cfg['data']['processed_dir']) / 'dataset.pkl', 'rb') as f:
    dm = pickle.load(f)
print('test samples:', len(dm.test_ds))

## 1 · Load trained models

Checkpoints saved by notebook 03 under `runs/<name>/model.pt`. Missing checkpoints are skipped,
not faked.

In [ ]:
RUNS = {
    'full':          ('runs/base/model.pt',          False, False),
    'no_graph':      ('runs/no_graph/model.pt',       True,  False),
    'single_task':   ('runs/single_task/model.pt',    False, False),
    'no_cross_attn': ('runs/no_cross_attn/model.pt',  False, True),
}

models = {}
for name, (path, drop_graph, no_cross) in RUNS.items():
    if not Path(path).exists():
        print(f'[skip] {name}: {path} not found — train it in notebook 03 first')
        continue
    mcfg = {'model': dict(cfg['model']), 'training': cfg['training'],
            'data': cfg['data']}
    mcfg['model']['no_cross_attn'] = no_cross
    model = MultimodalEnergyTransformer(mcfg).to(device)
    model.load_state_dict(torch.load(path, map_location=device))
    model._drop_graph = drop_graph
    model.eval()
    models[name] = model
    print(f'[ok]   {name}')

assert 'full' in models, 'the full model checkpoint is required for the comparison'

## 2 · Full model on the test year

Per-horizon MAE / RMSE / MAPE and **skill score vs seasonal-naive**. Generation is also reported
day-masked (07:00–18:00) because night-time PV is ~0 and otherwise flatters MAPE.

In [ ]:
full_results = evaluate_model(models['full'], dm, cfg, device, split='test')
print_table(full_results)
save_results(full_results, 'runs/base/test_metrics.json')

## 3 · Baselines

Three benchmarks on the identical split and windowing:
- **seasonal_naive** — persistence at lag 24 h / 168 h (no params; the skill-score reference).
- **DLinear** — Zeng et al. 2023, decomposition + linear, one per target (strong cheap baseline).
- **LightGBM** — gradient-boosted trees on calendar + weather + lag/rolling + building features
  (the strongest feature-engineered tabular baseline; ~800 trees per target×horizon).

In [ ]:
baselines = run_baselines(dm, cfg, split='test', device=device,
                          which=('seasonal_naive', 'dlinear', 'lightgbm'))
print('baselines computed:', list(baselines.keys()))

### Unified comparison table

Every predictor is scored with the **same** metric code, in physical kWh, on the same rows. This is
the table that answers the paper's claim: does the multimodal transformer beat a strong tabular
model and a strong linear model, at each horizon?

In [ ]:
from src.evaluate import _metrics
from src.data.dataset import DEMAND_CH, GEN_CH

horizons = list(cfg['data']['horizons'])
pred_full = predict_split(models['full'], dm, cfg, device, 'test')

# ground truth from the dataset index (physical units)
te = dm.test_ds
Yd = dm.demand_scaler.inverse_transform(
    np.array([[te.data[n, a+h, DEMAND_CH] for h in horizons] for (n,a) in te._index], dtype=np.float32).reshape(-1,1)
).reshape(-1, len(horizons))
Yg = dm.gen_scaler.inverse_transform(
    np.array([[te.data[n, a+h, GEN_CH] for h in horizons] for (n,a) in te._index], dtype=np.float32).reshape(-1,1)
).reshape(-1, len(horizons))

def score(target, pred):
    Y = Yd if target == 'demand' else Yg
    naive = baselines['seasonal_naive'][target + '_pred']
    rows = []
    for hi, h in enumerate(horizons):
        rows.append({'horizon': h, **_metrics(Y[:,hi], pred[:,hi], ref=naive[:,hi])})
    return pd.DataFrame(rows)

print('=== DEMAND: full model vs baselines ===')
demand_rows = {'Transformer (full)': pred_full['demand_pred']}
for b, d in baselines.items():
    demand_rows[b] = d['demand_pred']
demand_table = pd.concat({k: score('demand', v) for k, v in demand_rows.items()}, names=['model'])
print(demand_table.round(3).to_string())

print('\n=== GENERATION: full model vs baselines ===')
gen_rows = {'Transformer (full)': pred_full['gen_pred']}
for b, d in baselines.items():
    gen_rows[b] = d['gen_pred']
gen_table = pd.concat({k: score('gen', v) for k, v in gen_rows.items()}, names=['model'])
print(gen_table.round(3).to_string())

demand_table.to_csv('runs/base/table_demand.csv')
gen_table.to_csv('runs/base/table_gen.csv')

## 4 · Ablation comparison

Each ablation is scored against the full model. A stream earns its place only if removing it
measurably degrades the metrics.

In [ ]:
rows = []
for name, model in models.items():
    r = evaluate_model(model, dm, cfg, device, split='test')
    rows.append({
        'variant': name,
        'demand_MAE':   r['overall']['demand']['mae'],
        'demand_RMSE':  r['overall']['demand']['rmse'],
        'gen_MAE':      r['overall']['gen']['mae'],
        'gen_RMSE':     r['overall']['gen']['rmse'],
        'demand_skill': r['overall']['demand']['skill'],
        'gen_skill':    r['overall']['gen']['skill'],
    })
ablation_df = pd.DataFrame(rows).set_index('variant')
print(ablation_df.round(4).to_string())
ablation_df.to_csv('runs/base/table_ablations.csv')

# relative degradation of each ablation vs full
if 'full' in ablation_df.index:
    base = ablation_df.loc['full']
    print('\nDeterioration vs full (%, positive = worse):')
    print(((ablation_df - base) / base * 100).round(2).to_string())

## 5 · Error analysis

Break the full model's error down by horizon and by time-of-day to expose where it struggles.

In [ ]:
import matplotlib.pyplot as plt
hod_pred = (pred_full['anchor_hour'][:, None] + np.array(horizons)[None, :]) % 24
err_d = np.abs(pred_full['demand_pred'] - Yd)

fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
for hi, h in enumerate(horizons):
    by_hour = pd.Series(err_d[:, hi]).groupby(hod_pred[:, hi]).mean()
    ax[0].plot(by_hour.index, by_hour.values, marker='o', label=f'{h}h ahead')
ax[0].set_xlabel('hour of day'); ax[0].set_ylabel('mean abs demand error (kWh)')
ax[0].set_title('Demand error by time of day'); ax[0].legend()

mae_by_h = [np.mean(err_d[:, hi]) for hi in range(len(horizons))]
ax[1].bar([str(h) for h in horizons], mae_by_h, color='C0')
ax[1].set_xlabel('horizon (h)'); ax[1].set_ylabel('MAE (kWh)'); ax[1].set_title('Demand MAE by horizon')
plt.tight_layout(); plt.savefig('runs/base/fig_error_analysis.png', bbox_inches='tight'); plt.show()

### Reporting checklist
- [ ] Full-model per-horizon table (demand + gen + day-masked gen)
- [ ] Baseline comparison table (seasonal-naive / DLinear / LightGBM)
- [ ] Ablation table with % degradation vs full
- [ ] Error-by-time-of-day and error-by-horizon figures
- [ ] Every number traceable to a cell output above — none hand-entered